# Credit Events and Restructuring

**Purpose:** Show how to model three common distressed-credit workflows using pure Python — no external library dependencies.

**Prerequisites:** Basic familiarity with capital-structure priority, exchange offers, and liability management exercises.

**In this notebook:** We implement and run a recovery waterfall, compare hold-versus-tender economics for an exchange offer, and quantify an open-market repurchase LME.

> These are decision-support calculations, not mark-to-market valuations. The math is straightforward enough to live directly in a notebook.

## Concept

These analyses answer three common distressed-credit questions:

1. **Recovery waterfall** — who recovers what under a stressed distributable value, following the Absolute Priority Rule?
2. **Exchange offer** — is a hold-vs-tender trade-off economically attractive to holders?
3. **LME** — how much debt reduction and discount capture does a discounted liability-management move create?

In [ ]:
## Helpers

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional


def banner(title: str) -> None:
    print(f"\n{'=' * 72}")
    print(title)
    print("=" * 72)


def fmt_money(value: float) -> str:
    return f"${value:,.0f}"


def fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"

# Recovery Waterfall

# Claims are ordered by seniority. Each class recovers first from its pledged
# collateral (net of haircut); any shortfall rolls into the unsecured pool.
SENIORITY_ORDER = [
    "dip_financing",
    "administrative",
    "priority",
    "first_lien",
    "second_lien",
    "junior_secured",
    "senior_unsecured",
    "senior_subordinated",
    "subordinated",
    "mezzanine",
    "preferred_equity",
    "equity",
]


def execute_recovery_waterfall(
    total_value: float,
    claims: list[dict],
    allocation_mode: str = "pro_rata",
) -> dict:
    """Distribute `total_value` across `claims` following the Absolute Priority Rule.

    Each claim dict: seniority (str), principal (float), accrued (float, default 0),
    penalties (float, default 0), collateral_value (float, optional),
    haircut (float, default 0), id (str, optional), label (str, optional).
    """
    if allocation_mode != "pro_rata":
        raise ValueError("Only pro_rata allocation is supported in this notebook helper")

    enriched = []
    for i, c in enumerate(claims):
        seniority = c["seniority"]
        if seniority not in SENIORITY_ORDER:
            raise ValueError(f"Unknown seniority '{seniority}'")
        enriched.append({
            "id": c.get("id", f"claim-{i}"),
            "label": c.get("label", c.get("id", f"claim-{i}")),
            "seniority": seniority,
            "seniority_rank": SENIORITY_ORDER.index(seniority),
            "principal": c["principal"],
            "accrued": c.get("accrued", 0.0),
            "penalties": c.get("penalties", 0.0),
            "collateral_value": c.get("collateral_value"),
            "haircut": c.get("haircut", 0.0),
        })
    enriched.sort(key=lambda x: x["seniority_rank"])

    pool = total_value
    recoveries = []
    for c in enriched:
        total_claim = c["principal"] + c["accrued"] + c["penalties"]

        coll_recovery = 0.0
        deficiency = 0.0
        if c["collateral_value"] is not None:
            net_coll = c["collateral_value"] * (1.0 - c["haircut"])
            coll_recovery = min(net_coll, total_claim)
            deficiency = max(total_claim - coll_recovery, 0.0)
        else:
            deficiency = total_claim

        general_recovery = min(deficiency, pool)
        pool -= general_recovery

        total_recovery = coll_recovery + general_recovery
        recovery_rate = total_recovery / total_claim if total_claim > 0 else 0.0

        recoveries.append({
            "id": c["id"],
            "seniority": c["seniority"],
            "total_claim": total_claim,
            "collateral_recovery": coll_recovery,
            "general_recovery": general_recovery,
            "total_recovery": total_recovery,
            "recovery_rate": recovery_rate,
            "deficiency": max(total_claim - total_recovery, 0.0),
        })

    total_distributed = total_value - pool
    apr_satisfied = all(
        r["recovery_rate"] >= recoveries[i - 1]["recovery_rate"]
        if i > 0 and recoveries[i - 1]["recovery_rate"] < 1.0 else True
        for i, r in enumerate(recoveries)
    )

    return {
        "total_distributed": total_distributed,
        "residual": pool,
        "apr_satisfied": apr_satisfied,
        "per_claim_recovery": recoveries,
    }

In [ ]:
## Exchange Offer and LME Helpers

def analyze_exchange_offer(
    old_pv: float,
    new_pv: float,
    consent_fee: float = 0.0,
    equity_sweetener_value: float = 0.0,
    exchange_type: str = "par_for_par",
) -> dict:
    """Compare hold-vs-tender economics for a distressed exchange offer.

    old_pv: present value of the existing claim if not tendered.
    new_pv: present value of the new instrument received.
    consent_fee: cash consent / early-tender fee.
    equity_sweetener_value: estimated value of attached equity/warrants.
    exchange_type: one of par_for_par, discount, uptier, downtier.
    """
    valid_types = {"par_for_par", "discount", "uptier", "downtier"}
    canonical = exchange_type.lower().replace("-", "_")
    if canonical == "par":
        canonical = "par_for_par"
    if canonical not in valid_types:
        raise ValueError(f"Unknown exchange_type '{exchange_type}' — expected one of {valid_types}")
    if any(v < 0 for v in [old_pv, new_pv, consent_fee, equity_sweetener_value]):
        raise ValueError("old_pv, new_pv, consent_fee, equity_sweetener_value must be non-negative")

    tender_total = new_pv + consent_fee + equity_sweetener_value
    delta_npv = tender_total - old_pv
    breakeven_recovery = min(tender_total / old_pv, 1.0) if old_pv > 0 else 1.0
    tender_recommended = tender_total > old_pv * 1.02

    return {
        "exchange_type": canonical,
        "old_npv": old_pv,
        "new_npv": new_pv,
        "consent_fee": consent_fee,
        "equity_sweetener_value": equity_sweetener_value,
        "tender_total": tender_total,
        "delta_npv": delta_npv,
        "breakeven_recovery": breakeven_recovery,
        "tender_recommended": tender_recommended,
    }


def analyze_lme(
    lme_type: str,
    notional: float,
    repurchase_price_pct: float,
    opt_acceptance_pct: float = 1.0,
    ebitda: Optional[float] = None,
) -> dict:
    """Compute discount capture and leverage impact for an LME transaction.

    lme_type: open_market, tender_offer, amend_and_extend, or dropdown.
    notional: outstanding notional of the target instrument.
    repurchase_price_pct: price as fraction of par (0.60 = 60 cents).
    opt_acceptance_pct: fraction of holders participating (0.0–1.0).
    ebitda: if provided, leverage_impact block is included in the output.
    """
    if notional <= 0:
        raise ValueError(f"notional must be positive, got {notional}")
    if not 0.0 <= opt_acceptance_pct <= 1.0:
        raise ValueError(f"opt_acceptance_pct must be in [0.0, 1.0], got {opt_acceptance_pct}")

    kind = lme_type.lower().replace("-", "_").replace("&", "_")
    participating = notional * opt_acceptance_pct

    if kind in ("open_market", "open_market_repurchase", "omr"):
        if not 0.0 < repurchase_price_pct <= 1.5:
            raise ValueError(f"repurchase_price_pct must be in (0.0, 1.5], got {repurchase_price_pct}")
        par_retired = participating
        cost = par_retired * repurchase_price_pct
        canonical = "open_market_repurchase"
        remaining_holder_pct = 0.0
    elif kind in ("tender_offer", "tender"):
        if not 0.0 < repurchase_price_pct <= 1.5:
            raise ValueError(f"repurchase_price_pct must be in (0.0, 1.5], got {repurchase_price_pct}")
        par_retired = participating
        cost = par_retired * repurchase_price_pct
        canonical = "tender_offer"
        remaining_holder_pct = 0.0
    elif kind in ("amend_and_extend", "ae", "a_e"):
        if not 0.0 <= repurchase_price_pct <= 0.10:
            raise ValueError(f"extension_fee must be in [0.0, 0.10], got {repurchase_price_pct}")
        par_retired = 0.0
        cost = participating * repurchase_price_pct
        canonical = "amend_and_extend"
        remaining_holder_pct = 0.0
    elif kind == "dropdown":
        if not 0.0 <= repurchase_price_pct <= 1.0:
            raise ValueError(f"transferred-asset fraction must be in [0.0, 1.0], got {repurchase_price_pct}")
        par_retired = 0.0
        cost = 0.0
        canonical = "dropdown"
        remaining_holder_pct = repurchase_price_pct
    else:
        raise ValueError(f"Unknown lme_type '{lme_type}' — expected open_market, tender_offer, amend_and_extend, dropdown")

    discount_capture = par_retired - cost
    discount_capture_pct = discount_capture / par_retired if par_retired > 0 else 0.0

    result: dict = {
        "lme_type": canonical,
        "cost": cost,
        "notional_reduction": par_retired,
        "discount_capture": discount_capture,
        "discount_capture_pct": discount_capture_pct,
        "remaining_holder_impact_pct": remaining_holder_pct,
        "leverage_impact": None,
    }

    if ebitda and ebitda > 0:
        post_debt = notional - par_retired
        result["leverage_impact"] = {
            "pre_total_debt": notional,
            "post_total_debt": post_debt,
            "pre_leverage": notional / ebitda,
            "post_leverage": post_debt / ebitda,
            "leverage_reduction": (notional - post_debt) / ebitda,
        }

    return result

## Takeaways

- **`execute_recovery_waterfall()`** turns a stressed enterprise value into per-claim recoveries following the Absolute Priority Rule.
- **`analyze_exchange_offer()`** reframes a restructuring proposal in holder economics rather than issuer rhetoric.
- **`analyze_lme()`** measures debt reduction and leverage change from discounted liability-management moves.

These are pure Python helpers — the math is simple enough to embed directly in analysis notebooks without a compiled dependency.

In [ ]:
# Recovery Waterfall executable helper

SENIORITY_ORDER = [
    "dip_financing",
    "administrative",
    "priority",
    "first_lien",
    "second_lien",
    "junior_secured",
    "senior_unsecured",
    "senior_subordinated",
    "subordinated",
    "mezzanine",
    "preferred_equity",
    "equity",
]


def execute_recovery_waterfall(
    total_value: float,
    claims: list[dict],
    allocation_mode: str = "pro_rata",
) -> dict:
    """Distribute `total_value` across `claims` following the Absolute Priority Rule."""
    if allocation_mode != "pro_rata":
        raise ValueError("Only pro_rata allocation is supported in this notebook helper")

    enriched = []
    for i, c in enumerate(claims):
        seniority = c["seniority"]
        if seniority not in SENIORITY_ORDER:
            raise ValueError(f"Unknown seniority '{seniority}'")
        enriched.append({
            "id": c.get("id", f"claim-{i}"),
            "label": c.get("label", c.get("id", f"claim-{i}")),
            "seniority": seniority,
            "seniority_rank": SENIORITY_ORDER.index(seniority),
            "principal": c["principal"],
            "accrued": c.get("accrued", 0.0),
            "penalties": c.get("penalties", 0.0),
            "collateral_value": c.get("collateral_value"),
            "haircut": c.get("haircut", 0.0),
        })
    enriched.sort(key=lambda x: x["seniority_rank"])

    pool = total_value
    recoveries = []
    for c in enriched:
        total_claim = c["principal"] + c["accrued"] + c["penalties"]

        coll_recovery = 0.0
        deficiency = 0.0
        if c["collateral_value"] is not None:
            net_coll = c["collateral_value"] * (1.0 - c["haircut"])
            coll_recovery = min(net_coll, total_claim)
            deficiency = max(total_claim - coll_recovery, 0.0)
        else:
            deficiency = total_claim

        general_recovery = min(deficiency, pool)
        pool -= general_recovery

        total_recovery = coll_recovery + general_recovery
        recovery_rate = total_recovery / total_claim if total_claim > 0 else 0.0

        recoveries.append({
            "id": c["id"],
            "seniority": c["seniority"],
            "total_claim": total_claim,
            "collateral_recovery": coll_recovery,
            "general_recovery": general_recovery,
            "total_recovery": total_recovery,
            "recovery_rate": recovery_rate,
            "deficiency": max(total_claim - total_recovery, 0.0),
        })

    total_distributed = total_value - pool
    apr_satisfied = all(
        r["recovery_rate"] >= recoveries[i - 1]["recovery_rate"]
        if i > 0 and recoveries[i - 1]["recovery_rate"] < 1.0 else True
        for i, r in enumerate(recoveries)
    )

    return {
        "total_distributed": total_distributed,
        "residual": pool,
        "apr_satisfied": apr_satisfied,
        "per_claim_recovery": recoveries,
    }

In [ ]:
## Example Run

banner("Recovery waterfall")
claims = [
    {
        "id": "first_lien_tl",
        "label": "First Lien Term Loan",
        "seniority": "first_lien",
        "principal": 50_000_000.0,
        "accrued": 1_000_000.0,
        "collateral_value": 40_000_000.0,
        "haircut": 0.0,
    },
    {
        "id": "sr_unsec_notes",
        "label": "Senior Unsecured Notes",
        "seniority": "senior_unsecured",
        "principal": 80_000_000.0,
        "accrued": 2_000_000.0,
    },
    {
        "id": "sub_notes",
        "label": "Subordinated Notes",
        "seniority": "subordinated",
        "principal": 40_000_000.0,
        "accrued": 1_000_000.0,
    },
    {
        "id": "common_equity",
        "label": "Common Equity",
        "seniority": "equity",
        "principal": 0.01,
    },
]

waterfall = execute_recovery_waterfall(
    total_value=100_000_000.0,
    claims=claims,
    allocation_mode="pro_rata",
)
print(f"Total distributed: {fmt_money(waterfall['total_distributed'])}")
print(f"Residual         : {fmt_money(waterfall['residual'])}")
for row in waterfall["per_claim_recovery"]:
    print(
        f"  {row['id']:<20} claim={fmt_money(row['total_claim'])} "
        f"recovery={fmt_money(row['total_recovery'])} rate={fmt_pct(row['recovery_rate'])}"
    )

banner("Exchange offer")
exchange = analyze_exchange_offer(
    old_pv=45.0,
    new_pv=80.0,
    consent_fee=2.0,
    equity_sweetener_value=0.0,
    exchange_type="discount",
)
print(f"Tender total     : ${exchange['tender_total']:.2f}")
print(f"Delta NPV        : ${exchange['delta_npv']:+.2f}")
print(f"Breakeven recov. : {fmt_pct(exchange['breakeven_recovery'])}")
print(f"Tender?          : {exchange['tender_recommended']}")

banner("Open-market repurchase LME")
lme = analyze_lme(
    lme_type="open_market",
    notional=200_000_000.0,
    repurchase_price_pct=0.60,
    opt_acceptance_pct=0.40,
    ebitda=25_000_000.0,
)
print(f"Cash cost        : {fmt_money(lme['cost'])}")
print(f"Notional retired : {fmt_money(lme['notional_reduction'])}")
print(f"Discount capture : {fmt_money(lme['discount_capture'])}")
print(f"Holder impact    : {fmt_pct(lme['remaining_holder_impact_pct'])}")
if lme["leverage_impact"] is not None:
    leverage = lme["leverage_impact"]
    print(f"Pre leverage     : {leverage['pre_leverage']:.2f}x")
    print(f"Post leverage    : {leverage['post_leverage']:.2f}x")

# Summary for test runner
{
    "waterfall_apr_satisfied": waterfall["apr_satisfied"],
    "exchange_delta_npv": round(exchange["delta_npv"], 2),
    "lme_discount_capture": round(lme["discount_capture"], 2),
}